In [ ]:
%load_ext memory_profiler

In [ ]:
from esmdmsfunctions import get_simulation_results, gaussian_selection

%mprun -f get_simulation_results get_simulation_results(10, sel_func=gaussian_selection)

In [ ]:
%%writefile test_function.py

def test_function():
    test = 1000
    test_list = [i for i in range(10000)]

In [ ]:
%load_ext memory_profiler
from test_function import test_function

%mprun -f test_function test_function()

In [ ]:
%load_ext memory_profiler
from test_function import test_function

%mprun -f test_function test_function()

In [ ]:
from esmdmsfunctions import get_unique_df

In [ ]:
%load_ext memory_profiler

%mprun -f get_unique_df get_unique_df()

In [ ]:
whole_df = get_unique_df()

In [ ]:
whole_df

In [ ]:
from esmdmsfunctions import get_layer_dataframes

In [ ]:
%memit

In [ ]:
%load_ext memory_profiler
%memit

In [ ]:
from esmdmsfunctions import get_layer_dataframes, get_layer_dataframes_optimized, get_layer_dataframes_opt_2

In [ ]:
%memit

In [ ]:
layer_dfs_1 = get_layer_dataframes(["BF520"], layers=[0])

In [ ]:
%memit

In [ ]:
layer_dfs_2 = get_layer_dataframes_optimized(["BF520"], layers=[0])

In [ ]:
%memit

In [ ]:
layer_dfs_3 = get_layer_dataframes_opt_2(["BF520"], layers=[0])

In [ ]:
%memit

In [ ]:
layer_dfs_1[0]

In [ ]:
layer_dfs_2[0]

In [ ]:
layer_dfs_3[0]

In [ ]:
import sys

def deep_sizeof_df(df):
    total = sys.getsizeof(df)
    for col in df.columns:
        for val in df[col]:
            total += sys.getsizeof(val)
    return total / 1024**2

#layer_df = get_layer_dataframes(["BF520"], layers=[0])[0]
print(f"Shallow size: {layer_df[0].memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Deep size:    {deep_sizeof_df(layer_df[0]):.2f} MB")

In [ ]:
from esmdmsfunctions import get_layer_dataframes

In [ ]:
%memit

In [ ]:
import gc
import sys

# Show the largest objects still in memory
objects = [(sys.getsizeof(obj), type(obj)) for obj in gc.get_objects()]
objects.sort(key=lambda x: x[0], reverse=True)
for size, obj_type in objects[:10]:
    print(f"{size / 1024**2:.2f} MB — {obj_type}")

In [ ]:
%memit

In [ ]:
layer_dfs = get_layer_dataframes(["BF520"], layers=[0])

In [ ]:
%memit

In [ ]:
import sys

print(f"Size: {sys.getsizeof(layer_df) / 1024**2:.4f} MB")

In [ ]:
import sys

def deep_sizeof(obj, seen=None):
    size = sys.getsizeof(obj)
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)
    if isinstance(obj, dict):
        size += sum(deep_sizeof(k, seen) + deep_sizeof(v, seen) for k, v in obj.items())
    elif hasattr(obj, '__iter__') and not isinstance(obj, (str, bytes, bytearray)):
        size += sum(deep_sizeof(i, seen) for i in obj)
    return size

print(f"Size: {deep_sizeof(layer_df) / 1024**2:.4f} MB")

In [ ]:
layer_df[0].memory_usage(deep=True).sum() / 1024**2  # in MB

In [ ]:
%memit

In [ ]:
import gc
gc.collect()

In [ ]:
%memit

In [ ]:
import tracemalloc
import pickle
import numpy as np
from esmdmsfunctions import z_normalize


in_path = "/net/dali/home/barton/dhw28/popDMS/esmDMS/data/inference_results"

tracemalloc.start()

# Step 1: pickle load
layer_df = pickle.load(open(f"{in_path}/layer0/inference_df.pkl", 'rb'))
snapshot = tracemalloc.take_snapshot()
print(f"After pickle load: {sum(s.size for s in snapshot.statistics('filename')) / 1024**2:.2f} MB")

# Step 2: normalization
embeddings = np.array(layer_df["Embedding"].to_list())
snapshot = tracemalloc.take_snapshot()
print(f"After np.array():  {sum(s.size for s in snapshot.statistics('filename')) / 1024**2:.2f} MB")

for dim in range(embeddings.shape[1]):
    embeddings[:, dim] = z_normalize(embeddings[:, dim])
snapshot = tracemalloc.take_snapshot()
print(f"After normalize:   {sum(s.size for s in snapshot.statistics('filename')) / 1024**2:.2f} MB")

tracemalloc.stop()

In [ ]:
whole_df = get_unique_df()

In [ ]:
from esmdmsfunctions import get_layer_dataframes_opt_2, get_unique_df

In [ ]:
whole_df = get_unique_df()

In [ ]:
layer_dfs_3 = get_layer_dataframes_opt_2(["BF520"], layers=[0])

In [ ]:
whole_df.head()

In [ ]:
layer_dfs_3[0].head()

In [ ]:
df = layer_dfs_3[0].copy()

In [ ]:
df["Embedding"] = df["Embedding"].apply(tuple)

In [ ]:
# Collapse frequency counts by embedding
collapsed_df = df.groupby(["Embedding", "Generation", "Replicate"]).agg({
    "Frequency": "sum",
}).reset_index()
collapsed_df.head()


In [ ]:
len(collapsed_df)



In [ ]:
#def convert_collapsed_to_processed_df(collapsed_df):
processed_df = collapsed_df.copy()
n_reps = processed_df["Replicate"].nunique()
print(f"Number of replicates: {n_reps}")

#for rep in range(n_reps):
rep = 1
#test = processed_df[(processed_df["Generation"] == 0) & (processed_df["Replicate"] == rep)]


new_df = processed_df.copy()
for rep in range(1, n_reps+1):
    repprenums = processed_df[(processed_df["Generation"] == 0) & (processed_df["Replicate"] == rep)]
    new_column = f"Rep{rep}_PreNums"
    new_df[new_column] = new_df.apply(lambda row: repprenums.loc[
        (repprenums["Embedding"] == row["Embedding"]) &
        (repprenums["Replicate"] == rep),
        "Frequency"
    ].values[0] if ((row["Generation"] == 0) and (row["Replicate"] == rep)) else 0, axis=1)
    reppostnums = processed_df[(processed_df["Generation"] == 1) & (processed_df["Replicate"] == rep)]
    new_column = f"Rep{rep}_PostNums"
    new_df[new_column] = new_df.apply(lambda row: reppostnums.loc[
        (reppostnums["Embedding"] == row["Embedding"]) &
        (reppostnums["Replicate"] == rep),
        "Frequency"
    ].values[0] if ((row["Generation"] == 1) and (row["Replicate"] == rep)) else 0, axis=1)
    
        


    

In [ ]:
new_df.drop(columns=["Frequency", "Generation", "Replicate"], inplace=True)

In [ ]:
new_df

In [ ]:
# Collect by embedding, summing rep1_prenums and rep1_postnums
final_df = new_df.groupby("Embedding").agg({
    "Rep1_PreNums": "sum",
    "Rep1_PostNums": "sum",
    "Rep2_PreNums": "sum",
    "Rep2_PostNums": "sum",
    "Rep3_PreNums": "sum",
    "Rep3_PostNums": "sum",
}).reset_index()



In [ ]:
import numpy as np
final_df["Embedding"] = final_df["Embedding"].apply(np.array)

In [ ]:
final_df

In [ ]:
import numpy as np
from esmdmsfunctions import get_layer_dataframes, name_to_path
import pickle
import pandas as pd


def process_layer_to_final_df(name, layer):
    layer_dfs = get_layer_dataframes([name], layers=[layer], normalize=False)
    df = layer_dfs[0].copy()
    df["Embedding"] = df["Embedding"].apply(tuple)
    # Collapse frequency counts by embedding
    collapsed_df = df.groupby(["Embedding", "Generation", "Replicate"]).agg({
        "Frequency": "sum",
    }).reset_index()
    
    processed_df = collapsed_df.copy()
    n_reps = processed_df["Replicate"].nunique()

    new_df = processed_df.copy()
    for rep in range(1, n_reps+1):
        repprenums = processed_df[(processed_df["Generation"] == 0) & (processed_df["Replicate"] == rep)]
        new_column = f"Rep{rep}_PreNums"
        new_df[new_column] = new_df.apply(lambda row: repprenums.loc[
            (repprenums["Embedding"] == row["Embedding"]) &
            (repprenums["Replicate"] == rep),
            "Frequency"
        ].values[0] if ((row["Generation"] == 0) and (row["Replicate"] == rep)) else 0, axis=1)
        reppostnums = processed_df[(processed_df["Generation"] == 1) & (processed_df["Replicate"] == rep)]
        new_column = f"Rep{rep}_PostNums"
        new_df[new_column] = new_df.apply(lambda row: reppostnums.loc[
            (reppostnums["Embedding"] == row["Embedding"]) &
            (reppostnums["Replicate"] == rep),
            "Frequency"
        ].values[0] if ((row["Generation"] == 1) and (row["Replicate"] == rep)) else 0, axis=1)

    new_df.drop(columns=["Frequency", "Generation", "Replicate"], inplace=True)
    
    # Collect by embedding, summing rep1_prenums and rep1_postnums
    final_df = new_df.groupby("Embedding").agg({
        col: "sum" for col in new_df.columns if col.startswith("Rep")
    }).reset_index()

    final_df["Embedding"] = final_df["Embedding"].apply(np.array)


    # Save the final_df to a pickle file
    out_path = name_to_path(name)
    final_df.to_pickle(f"{out_path}/layer{layer}/final_df.pkl")


In [ ]:
for layer in range(31):
    print(f"Processing layer {layer}...")
    process_layer_to_final_df("BG505", layer)

In [ ]:
from esmdmsfunctions import name_to_path
import pandas as pd
import pickle

def make_sim_df(name, layer):
    """Create an optimized df to input into the simulaiton"""
    name_path = name_to_path(name)
    final_df_path = f"{name_path}/layer{layer}/final_df.pkl"

    final_df = pd.read_pickle(final_df_path)
    
    n_reps = len(final_df.columns) // 2
    
    col_drops = [f"Rep{rep+1}_PostNums" for rep in range(n_reps)]
    final_df = final_df.drop(columns=col_drops)
    
    
    final_df.to_pickle(f"{name_path}/layer{layer}/sim_df.pkl")
    

In [ ]:
n_layers = 31
names = ["BF520", "BG505"]
for layer in range(n_layers):
    for name in names:
        print(f"Making sim df for name: {name} and layer: {layer}")
        make_sim_df(name, layer)

In [ ]:
%load_ext memory_profiler

In [ ]:
%memit

In [ ]:
# Ok, but seriously, what am I doing? Let's set up the simulation scripts to be able to run for a layer in a batch. Current goal is to 
# 1. Run BF ands BG in simulation at the same time.
#   FUCK, I already normalized the embeddings. I think I need tgo preprocess a combined one actually.
# 2. Get gamma normalization correlation data between simulated and inferred 
# Ok. Let's create the combined data.




In [ ]:

from pathlib import Path

def process_layer_to_final_df_combined(names, layer, comb_dir):
    layer_dfs = get_layer_dataframes(names, layers=[layer], normalize=False)
    df = layer_dfs[0].copy()
    df["Embedding"] = df["Embedding"].apply(tuple)

    # Collapse frequency counts by embedding
    collapsed_df = df.groupby(["Embedding", "Generation", "Replicate"], as_index=False).agg(
        Frequency=("Frequency", "sum")
    )

    # Pivot: rows=Embedding, columns=(Replicate, Generation), values=Frequency
    # This replaces the entire per-replicate apply loop
    pivot = collapsed_df.pivot_table(
        index="Embedding",
        columns=["Replicate", "Generation"],
        values="Frequency",
        fill_value=0,
    )

    # Rename (rep, gen) -> Rep{rep}_PreNums / Rep{rep}_PostNums
    pivot.columns = [
        f"Rep{rep}_PreNums" if gen == 0 else f"Rep{rep}_PostNums"
        for rep, gen in pivot.columns
    ]
    pivot.columns.name = None
    final_df = pivot.reset_index()
    final_df["Embedding"] = final_df["Embedding"].apply(np.array)

    out_dir = Path(comb_dir) / f"layer{layer}"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    
    final_df.to_pickle(out_dir / "final_df.pkl")

In [ ]:
names = ["BF520", "BG505"]
n_layers = 31

from pathlib import Path

out_dir = Path(name_to_path(name)) / f"layer{layer}"
out_dir.mkdir(parents=True, exist_ok=True)
final_df.to_pickle(out_dir / "final_df.pkl")

for layer in range(n_layers):
    

## SOME PLOTS

In [ ]:
import esmdmsfunctions
from esmdmsfunctions import *
import pickle

In [ ]:
layer_path = "/net/dali/home/barton/dhw28/popDMS/esmDMS/esm_sim_saves/combined_sim_layer"

In [ ]:
%load_ext memory_profiler
%memit

In [ ]:
all_layer_data = {col: {} for col in range(5)}

for layer in range(31):
    with open(f"{layer_path}{layer}.pkl", "rb") as f:
        layer_data = pickle.load(f)
    for col in range(5):
        all_layer_data[col][layer] = layer_data[col]

In [ ]:
%memit

In [ ]:
plot_fitness_over_time(all_layer_data)

In [ ]:
plot_inferred_vs_true_sel(all_layer_data)

In [ ]:
for layer in range(31):
    comp_inf_vs_real_fits(all_layer_data, layer)

In [ ]:
for layer in range(31):
    comp_inf_vs_real_fits_rank(all_layer_data, layer)

In [ ]:
both_data = {}

for layer in range(20):
    with open(f"{layer_path}{layer}.pkl", "rb") as f:
        layer_data = pickle.load(f)
        both_data[layer] = layer_data

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as st


num_reps = 6
rep_combs = []
for i in range(num_reps):
    for j in range(i+1, num_reps):
        rep_combs.append((i, j))
num_combs = len(rep_combs)
print(f"rep_combs: {rep_combs}")

# Square, diagonal plots
fig, axs = plt.subplots(num_reps, num_reps, figsize=(4*num_reps, 4*num_reps))
for i in range(num_reps):
    for j in range(num_reps):
        ax = axs[i, j]
        """if i == j:
            
            
        else:"""
        
        best_pearson_r = -2
        total_pearson_r = 0
        
        
        for layer in range(len(both_data)):
            s_i = both_data[layer][2][0][i]
            s_j = both_data[layer][2][0][j]
            
            s_i = s_i / np.max(np.abs(s_i))
            s_j = s_j / np.max(np.abs(s_j))
            
            #print(f"s_i shjape: {s_i.shape}, s_j shape: {s_j.shape}")

            pearson_r = st.pearsonr(s_i, s_j)[0]

            total_pearson_r += pearson_r
            if pearson_r > best_pearson_r:
                best_pearson_r = pearson_r
            
            ax.scatter(s_i, s_j, label=f'Layer {layer}', alpha=0.6)
        avg_pearson_r = total_pearson_r / len(both_data)
        ax.set_title(f'Best r: {best_pearson_r:.2f}, Avg r: {avg_pearson_r:.2f}', fontsize=12)
            
        #ax.set_xlabel(f'Select. Coeff. Rep {i+1}')
        #ax.set_ylabel(f'Select. Coeff. Rep {j+1}')
        ax.axis('square')
plt.style.use('seaborn-v0_8-darkgrid')
plt.suptitle('Replicate Consistency Plots Across Layers (BG505 + BF520)', fontsize=48,
                y=0.92)

# Label the X and Y axes of the outer plots


rep_to_label = {
    1: 'BF Rep 1',
    2: 'BF Rep 2',
    3: 'BF Rep 3',
    4: 'BG Rep 1',
    5: 'BG Rep 2',
    6: 'BG Rep 3'
}

for i in range(num_reps):
    axs[num_reps-1, i].set_xlabel(rep_to_label[i+1], fontsize=24)
    axs[i, 0].set_ylabel(rep_to_label[i+1], fontsize=24)

# change the font of the plot to latex and bold
plt.rcParams.update({'font.family': 'serif',
                    'text.usetex': True,
                    'font.weight': 'bold'})
                    
plt.show()

In [ ]:
both_data[0][2][0][1]